# Research Question Formation

## Project Scope

This project analyzes the **BatteryLife dataset**, a large-scale benchmark dataset introduced in the KDD 2025 Datasets and Benchmarks Track for battery life prediction. The dataset contains multivariate time-series measurements from approximately 1,000 batteries collected from multiple experimental datasets. Each battery consists of a sequence of charge-discharge cycles containing measurements such as voltage, current, and discharge capacity.

Exploratory Data Analysis (EDA) performed in Project Checkpoint 1 revealed several key properties of the dataset:

- **Large variability in cycle life:** Batteries exhibit a wide range of total cycle life values, indicating diverse degradation behaviors.

- **Variable-length time series:** Different batteries contain different numbers of cycles, which introduces challenges for modeling sequential data.

- **Nonlinear degradation behavior:** Battery capacity degradation is typically nonlinear, with slow early degradation followed by accelerated decline near end-of-life.

- **Variation in initial capacity:** Initial battery capacities vary across batteries due to manufacturing differences and experimental conditions.

Based on these observations, this project focuses on predicting the **Remaining Useful Life (RUL)** of batteries using early-cycle measurements. The analysis will investigate both classical machine learning approaches and sequence-based deep learning models to determine how well early-cycle measurements can predict battery lifetime.


## Research Questions

### RQ1
**How accurately can tree-based regression models predict battery cycle life using early-cycle capacity and voltage features?**

- **Data mining task:** Supervised regression for predicting continuous battery cycle life.
- **Algorithms:** Regression Trees and Random Forest Regression to model nonlinear relationships between early-cycle measurements and battery lifetime.
- **Evaluation criteria:** Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) will be used to evaluate prediction accuracy and compare model performance.

### RQ2
**How does the amount of early-cycle data used for prediction affect the accuracy of remaining useful life estimation?**

- **Data mining task:** Supervised regression with varying feature windows based on the number of early cycles used.
- **Algorithms:** k-Nearest Neighbor Regression and Random Forest Regression to evaluate how prediction accuracy changes with additional early-cycle information.
- **Evaluation criteria:** MAE and RMSE will be measured while varying the number of cycles used as input features.

### RQ3
**Do sequence-based deep learning models improve battery life prediction accuracy compared to classical machine learning models?**

- **Data mining task:** Sequential time-series modeling of battery degradation behavior.
- **Algorithms:** Long Short-Term Memory (LSTM) neural networks to capture temporal dependencies in battery degradation sequences.
- **Evaluation criteria:** MAE and RMSE will be used to compare the predictive performance of LSTM models against classical regression baselines.


## RQ-to-Method Mapping

| RQ | Data Mining Task | Algorithms | Evaluation Metrics |
|----|------------------|-----------|-------------------|
| RQ1 | Supervised regression for battery life prediction | Regression Trees, Random Forest Regression | MAE, RMSE |
| RQ2 | Prediction using different early-cycle feature windows | k-Nearest Neighbor Regression, Random Forest Regression | MAE, RMSE |
| RQ3 | Sequential time-series modeling of battery degradation | Long Short-Term Memory (LSTM) neural networks | MAE, RMSE |


## Motivation and Feasibility

**Motivation:**  
Exploratory data analysis revealed large variability in battery cycle life and nonlinear degradation behavior. Early-cycle capacity measurements appear to contain signals that may help predict long-term battery life, motivating the use of predictive models.

**Non-triviality:**  
Battery degradation is sequential and nonlinear, meaning that simple models using only static features may miss temporal patterns present in the degradation curves. Investigating both classical regression models and sequence-based models allows us to study whether temporal modeling provides additional predictive power.

**Feasibility:**  
The dataset size is manageable for both classical machine learning models and deep learning models. Libraries such as scikit-learn and PyTorch provide implementations that make these algorithms accessible for experimentation.

**Risks:**  
Potential challenges include variability across batteries, sensitivity to feature selection, and computational costs associated with training deep learning models. Careful model validation and parameter tuning will be required to mitigate these risks.


## Methodological Planning

**Algorithms:**  
The analysis will begin with classical regression algorithms including Regression Trees, Random Forest Regression, and k-Nearest Neighbor Regression. These models provide interpretable baselines for predicting battery cycle life from early-cycle measurements.

**External Method:**  
A Long Short-Term Memory (LSTM) neural network will be implemented to model sequential degradation patterns across battery cycles.

**Evaluation:**  
Models will be evaluated using regression metrics including Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE). These metrics provide interpretable measures of prediction accuracy and allow comparison across models.

**Baselines:**  
Random Forest Regression will serve as the primary baseline model due to its strong performance on structured tabular datasets.

**Initial Feasibility Testing:**  
Preliminary experiments will be performed to confirm that early-cycle features can be extracted and used to train baseline models. These experiments will also verify that the dataset structure supports sequential modeling approaches.


## Initial Method Runs for Feasibility

To ensure that the proposed analytical methods are feasible, a set of small preliminary experiments is performed using the project dataset. These tests are designed to verify that the necessary Python libraries and algorithms function correctly and can be applied to the battery dataset.

First, a simple feature is extracted from the early cycles of each battery by computing the average discharge capacity from the first few cycles. This feature is then used to test machine learning algorithms that will be used in later stages of the project.

The purpose of these experiments is not to produce final results, but rather to confirm that the modeling tools and packages operate correctly and that the dataset can support the proposed prediction tasks.


In [6]:
import json
import pandas as pd

with open("Dataset/Life labels/total_MICH_labels.json", "r") as f:
    mich_labels = json.load(f)

life_df = pd.DataFrame(
    list(mich_labels.items()),
    columns=["battery_id", "cycle_life"]
)

print(life_df.head())


                                     battery_id  cycle_life
0   MICH_BLForm9_pouch_NMC_45C_0-100_1-1C_i.pkl         191
1  MICH_MCForm21_pouch_NMC_25C_0-100_1-1C_a.pkl         362
2  MICH_MCForm34_pouch_NMC_45C_0-100_1-1C_d.pkl         457
3  MICH_MCForm35_pouch_NMC_45C_0-100_1-1C_e.pkl         351
4  MICH_BLForm12_pouch_NMC_25C_0-100_1-1C_b.pkl         289


In [10]:
import os
import numpy as np

mich_path = "Dataset/MICH"

early_caps = []
battery_ids = []

for file in os.listdir(mich_path):

    if file.endswith(".pkl"):
        battery = pd.read_pickle(os.path.join(mich_path, file))
        cycles = battery["cycle_data"][:5]

        caps = []
        for cycle in cycles:
            caps.append(max(cycle["discharge_capacity_in_Ah"]))

        early_caps.append(np.mean(caps))
        battery_ids.append(file)   # <-- FIXED

feature_df = pd.DataFrame({
    "battery_id": battery_ids,
    "early_capacity": early_caps
})


In [11]:
data = pd.merge(feature_df, life_df, on="battery_id")

print("Merged dataset size:", data.shape)
data.head()


Merged dataset size: (40, 3)


,battery_id,early_capacity,cycle_life
0,MICH_MCForm40_pouch_NMC_45C_0-100_1-1C_j.pkl,2.3496,392
1,MICH_BLForm14_pouch_NMC_25C_0-100_1-1C_d.pkl,2.2134,325
2,MICH_BLForm4_pouch_NMC_45C_0-100_1-1C_d.pkl,2.3408,298
3,MICH_MCForm23_pouch_NMC_25C_0-100_1-1C_c.pkl,2.2186,390
4,MICH_MCForm37_pouch_NMC_45C_0-100_1-1C_g.pkl,2.3484,404


### Feasibility Test: Random Forest Regression

To verify that tree-based regression models can be successfully applied to the dataset, a Random Forest regression model is trained using the extracted early-cycle feature.

Random Forests are chosen because they are robust to nonlinear relationships and are widely used for tabular prediction tasks. This model serves as a simple baseline to confirm that machine learning algorithms can be trained using the extracted battery features and that the scikit-learn implementation works correctly within the project environment.

The purpose of this test is not to obtain final predictive results, but to confirm that the modeling pipeline is functional and that tree-based regression models can be applied to the dataset.


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = data[["early_capacity"]]
y = data["cycle_life"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

print("Random Forest feasibility test R^2:", rf.score(X_test, y_test))


Random Forest feasibility test R^2: 0.13678182184558696


### Feasibility Test: k-Nearest Neighbor Regression

In addition to tree-based models, a k-Nearest Neighbor (k-NN) regression model is tested to ensure that distance-based learning algorithms can also be applied to the dataset.

k-NN regression predicts battery cycle life based on the similarity between batteries in the feature space. Testing this model confirms that alternative regression approaches can be implemented using the extracted features and that the dataset supports different types of machine learning methods.

Similar to the previous experiment, this run serves only as a feasibility check to verify that the algorithm and supporting libraries function correctly.


In [14]:
from sklearn.neighbors import KNeighborsRegressor

# Initialize k-NN model
knn = KNeighborsRegressor(n_neighbors=5)

# Train model
knn.fit(X_train, y_train)

# Evaluate model
knn_score = knn.score(X_test, y_test)

print("k-NN feasibility test R^2:", knn_score)


k-NN feasibility test R^2: 0.029001659687187376


## Collaboration Declaration

On my honor, I declare the following resources were used in completing this assignment:

**Collaborators:**  
None. 

**Web Sources:**  
I consulted official documentation for the following libraries to understand usage and syntax:
- scikit-learn documentation (https://scikit-learn.org/)
- pandas documentation (https://pandas.pydata.org/)
- matplotlib documentation (https://matplotlib.org/)

**AI Tools:**  
I used ChatGPT as an AI assistant to help with brainstorming research questions, structuring explanations, and debugging code. All code and written explanations were reviewed and edited by me to ensure correctness and understanding.